# Coupled mechanochemical model of nuclear mechanosensing on nanopillars - minimal

Here, we implement the model originally presented in [Sun et al. 2016, Biophysical Journal](https://www.sciencedirect.com/science/article/pii/S0006349516302430?via%3Dihub) and then expanded by [Scott et al. 2021, PNAS](https://www.pnas.org/doi/10.1073/pnas.2021571118). The goal is to describe the nuclear translocation of the mechanosensitive transcriptional regulators YAP/TAZ, given some mechanical input from the substrate. 

Here, we specifically consider cell geometries on substrates with cylindrical nanopillar structures, where the nuclear shape is allowed to evolve over time due to a downward force attributed to the perinuclear actin cap.

The geometry in this model is divided into 4 domains - two volumes and two surfaces:
- plasma membrane (PM)
- Cytosol (Cyto)
- Nuclear envelope (NE)
- Interior of the nucleus (Nuc)

This model has 18 species, including one PM species:
* RhoA_GTP: activated form of RhoA localized to the PM

The cytosolic species include:
* pFAK: phosphorylated FAK (unphosphorylated accounted for by mass conservation)
* RhoA_GDP: inactive form of RhoA
* ROCK_A: activated Rho-associated protein kinase
* mDia_A: activated version of the formin mDia
* Myo_A: activated myosin (associated with stress fiber formation)
* LIMK_A: activated LIM kinase
* Cofilin_NP: nonphosphorylated (activated) cofilin
* FActin: polymerized form of actin
* GActin: monomeric form of actin
* YAPTAZ: dephosphorylated form of YAP/TAZ in the cytosol
* YAPTAZ_seq: phosphorylated form of YAP/TAZ (preferentially remains cytosolic)

The nuclear membrane species include:
* eta: density of nuclear membrane per unit NE surface
* LaminA: dephosphorylated form of Lamin A, a structural component of the nuclear lamina
* LaminAp: phosphorylated form of Lamin A
* NPC_A: activated (open) nuclear pore complexes
* NPC: inactivated (closed) nuclear pore complexes

Finally, the nucleus contains:
* YAPTAZ_nuc: concentration of YAP/TAZ in the nucleus

In [1]:
import dolfin as d
import sympy as sym
import numpy as np
import pathlib
import logging
import argparse
import matplotlib.pyplot as plt


from smart import config, mesh, model, mesh_tools
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    sbmodel_from_locals,
)
import sys
from scipy.optimize import fsolve

try:
    import ufl_legacy as ufl
except ImportError:
    import ufl

from mech_parser_args import add_coupled_arguments
here = pathlib.Path.cwd()
sys.path.insert(0, (here / ".." / "scripts").as_posix())

smart_logger = logging.getLogger("smart")
smart_logger.setLevel(logging.DEBUG)
logger = logging.getLogger("mechanotransduction")
logger.setLevel(logging.INFO)
logger.info("Starting mechanotransduction example")
sys.path.insert(0, "/root/shared/gitrepos/smart-mechanotransduction/mesh-files")
import spread_cell_mesh_generation as mesh_gen

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
ROOT -2026-09-13 15:40:53,127 mechanotransduction - INFO - Starting mechanotransduction example (3298999447.py:35)


If running a series of tests, we use `argparse` to read in the test conditions. Otherwise, we load default parameters to the `args` dict.

In [2]:
parser = argparse.ArgumentParser()
add_coupled_arguments(parser)
try:
    shell = get_ipython().__class__.__name__
    args = {}
    if shell == 'ZMQInteractiveShell': # then running a jupyter notebook
        args["time_step"] = 0.01
        args["a0_npc"] = 1.0
        cur_dir = str(pathlib.Path.cwd() / "..")
        # args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h1.0_p2.5_r0.25_cellRad17.45")
        # args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h3.0_p3.5_r0.5_cellRad15.5")
        args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h1.0_p1.0_r0.1_cellRad16.55")
        args["outdir"] = pathlib.Path(f"/root/scratch/test_h2.5")
        args["force_val"] = 500.0
        args["t0_deform"] = 1000.0
        args["actin_boost"] = 30.0
        args["lamin_diff"] = 0.0
        args["lamin_abundance"] = 1.0
        args["k_phos"] = 0.001
        args["phiE"] = 0.5
        args["nanopillar_radius"] = 0.2
        args["nanopillar_height"] = 2.5
        args["nanopillar_spacing"] = 3.0
except:
    args = {}
if len(args) == 0: # then most likely running as a script
    args = vars(parser.parse_args())

timer = d.Timer("mechanotransduction-example")
incl_pore = False
no_substrate = False
laminOnly = False

Now we define units for use in the model.

In [3]:
# Aliases - base units
uM = unit.uM
um = unit.um
molecule = unit.molecule
sec = unit.sec
dimensionless = unit.dimensionless
# Aliases - units used in model
D_unit = um**2 / sec
flux_unit = uM * um / sec
vol_unit = uM
surf_unit = molecule / um**2
# stiffness units
kPa = unit.kilopascal

# Model generation

For each step of model generation, refer to SMART Example 3 or API documentation for further details.

We first define compartments and the compartment container. Note that we can specify nonadjacency for surfaces in the model, which is not required, but can speed up the solution process.

In [4]:
# Cyto = Compartment("Cyto", 3, um, 1,
#                    deform=["0","0","0"], manual_update = True)
# Contact = Compartment("Contact", 2, um, 11, deform=["0","0","0"])
# PM = Compartment("PM", 2, um, 10, deform=["0","0","0"], manual_update = True)
Nuc = Compartment("Nuc", 3, um, 2,
                  deform=["0","0","0"], manual_update = True)
R0 = 4.1
z0 = 1.04
NE = Compartment("NE", 2, um, 10,
                 deform=["0","0","0"], manual_update = True,
                 normals = [f"x[0]/sqrt(pow(x[0],2) + pow(x[1],2) + pow((x[2]-{R0+z0}),2))",
                            f"x[1]/sqrt(pow(x[0],2) + pow(x[1],2) + pow((x[2]-{R0+z0}),2))",
                            f"(x[2]-{R0+z0})/sqrt(pow(x[0],2) + pow(x[1],2) + pow((x[2]-{R0+z0}),2))"])
# PM.specify_nonadjacency(["NE", "Nuc"])
# Contact.specify_nonadjacency(["NE", "Nuc"])
# NE.specify_nonadjacency(["Contact", "PM"])

Now we initialize all species in the model.

In [ ]:
if args["a0_npc"] > 0:
    # aNPC = Species("aNPC", 0.0, dimensionless, 0.0, D_unit, "NE")
    aNPC = Parameter.mesh_quantity("aNPC", 0.0, dimensionless, compartment="NE")
init_vals = [0.3, 1.0, 33.6, 0.0, 0.01, 1.5, 0.1, 1.8, 17.9, 482.4, 0.0, 0.0, 0.7, 0.2, 0.7, 0.0]
sp_names = ["pFAK", "RhoA_GDP", "RhoA_GTP", "ROCK_A", "mDia_A", "Myo_A", "LIMK_A", "Cofilin_NP",
            "FActin", "GActin", "LaminA", "NPC_A", "YAPTAZ", "YAPTAZ_seq", "YAPTAZ_nuc"]
init_vec = init_vals

# GActin = Species("GActin", init_vec[9], vol_unit, 13.37, D_unit, "Cyto")
# FActin = Species("FActin", init_vec[8], vol_unit, 0.6, D_unit, "Cyto")
# additional settings!
actinBoost = args["actin_boost"]
actinDecay = 0.5
laminDiff = args["lamin_diff"]
laminAbundance = args["lamin_abundance"]
nanopillar_rad = args["nanopillar_radius"]
nanopillar_height = args["nanopillar_height"]
nanopillar_spacing = args["nanopillar_spacing"]
# if nanopillar_rad == 0.0:
#     FActin0 = 140
# elif nanopillar_rad <= 0.15:
#     FActin0 = 80.0
# elif nanopillar_rad <= 0.25:
#     FActin0 = 100.0
# elif nanopillar_rad <= 0.5:
#     FActin0 = 120.0
# else:
#     FActin0 = 140.0
FActin0 = 110.0

LaminA = Species("LaminA", init_vec[10], surf_unit, 0.0, D_unit, "NE",
                  alt_deform=["0.0","0.0","0.0"], alt_manual_update=True)
if not laminOnly:
    NPC_A = Species("NPC_A", init_vec[11], surf_unit, 0.0, D_unit, "NE")
    # YAPTAZ = Species("YAPTAZ", init_vec[12], vol_unit, 80.0, D_unit, "Cyto")  # non-phosphorylated in the cytosol
    # YAPTAZ_seq = Species("YAPTAZ_seq", init_vec[13], vol_unit, 4.0, D_unit, "Cyto")  # seq in the cytosol
    YAPTAZ_nuc = Species("YAPTAZ_nuc", init_vec[14], vol_unit, 4.0, D_unit, "Nuc")

 2026-09-13 15:40:53,179 smart.model_assembly - DEBUG - Initializing parameter aNPC as mesh quantity (model_assembly.py:743) 


Now we define reactions and parameters across 3 modules.

Module A: Substrate stiffness -> initial signaling events

Initialize nuclear geometry in cytosol (smallest force that allows it to fit). This can take a long time...

In [6]:
import nuclear_deformation_mixed3d as nuc_def
phiE = args["phiE"]
nuc_args = dict()
nuc_args["nuc_only"] = True
nuc_args["mesh_folder"] = args["mesh_folder"]
nuc_args["max_force"] = 0.0 # only for initial phase
nuc_args["start_force"] = 0.0
nuc_args["softFactor"] = (1-phiE) + phiE*laminAbundance
nuc_args["bulk_mod"] = 1e8 * nuc_args["softFactor"]
nuc_args["nanopillar_radius"] = nanopillar_rad
nuc_args["nanopillar_height"] = nanopillar_height
nuc_args["nanopillar_spacing"] = nanopillar_spacing
nuc_args["outdir"] = args["outdir"] / "mech_data"
nuc_args["contactRad"] = 20.0

nuc_dict = nuc_def.start_nuc_mech(nuc_args)

ROOT -2026-09-13 15:40:53,213 mechanotransduction - INFO - Starting nuclear mechanics calculation (nuclear_deformation_mixed3d.py:21)


ROOT -2026-09-13 15:40:55,550 mechanotransduction - INFO - Starting force is 0.0 (nuclear_deformation_mixed3d.py:461)
ROOT -2026-09-13 15:40:55,550 mechanotransduction - INFO - Starting pressure is 0.0 (nuclear_deformation_mixed3d.py:462)
ROOT -2026-09-13 15:41:02,060 mechanotransduction - INFO - Current volume is 61.80048529445672 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)
ROOT -2026-09-13 15:41:02,062 mechanotransduction - INFO - Current force is 0.0 (nuclear_deformation_mixed3d.py:589)
ROOT -2026-09-13 15:41:02,063 mechanotransduction - INFO - Current pressure is 0.0 (nuclear_deformation_mixed3d.py:590)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 1.034e-11 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton solver finished in 0 iterations and 0 linear solver iterations.
Building point search tree to accelerate distance queries.
Computed bounding box tree with 5487 nodes for 2744 points.
Solving nonlinear variational problem.


ROOT -2026-09-13 15:41:03,192 mechanotransduction - INFO - Done computing idx = 1 after 1 iterations, def = -0.00011254376287879507 (nuclear_deformation_mixed3d.py:738)


  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 0: r (abs) = 3.300e+04 (tol = 1.000e-06) r (rel) = 1.000e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 1: r (abs) = 3.032e+00 (tol = 1.000e-06) r (rel) = 9.188e-05 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 2: r (abs) = 1.791e-02 (tol = 1.000e-06) r (rel) = 5.427e-07 (tol = 1.000e-06)
  Newton solver finished in 2 iterations and 2 linear solver iterations.
Current min Jacobian is 0.9998363171614993


...

Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 4.342e+04 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 3.965e+00 (tol = 1.000e-08) r (rel) = 9.131e-05 (tol = 1.000e-06)
  Newton iteration 2: r (abs) = 1.456e-05 (tol = 1.000e-08) r (rel) = 3.354e-10 (tol = 1.000e-06)
  Newton solver finished in 2 iterations and 2 linear solver iterations.
Solving nonlinear variational problem.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 0: r (abs) = 3.990e+00 (tol = 1.000e-06) r (rel) = 1.000e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 1: r (abs) = 2.807e+02 (tol = 1.000e-06) r (rel) = 7.035e+01 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Ne

ROOT -2026-09-13 15:41:51,535 mechanotransduction - INFO - Current volume is 142.6029728912359 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)
ROOT -2026-09-13 15:41:51,541 mechanotransduction - INFO - Done computing idx = 7 after 6 iterations, def = 1.3509697603966622 (nuclear_deformation_mixed3d.py:738)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 3.537e+04 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 2.625e+00 (tol = 1.000e-08) r (rel) = 7.424e-05 (tol = 1.000e-06)
  Newton iteration 2: r (abs) = 6.278e-06 (tol = 1.000e-08) r (rel) = 1.775e-10 (tol = 1.000e-06)
  Newton solver finished in 2 iterations and 2 linear solver iterations.


ROOT -2026-09-13 15:41:51,969 mechanotransduction - INFO - Current volume is 142.602972891236 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 6.278e-06 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 1.343e-09 (tol = 1.000e-08) r (rel) = 2.140e-04 (tol = 1.000e-06)
  Newton solver finished in 1 iterations and 1 linear solver iterations.


In [7]:
deformed_mesh_file = args["outdir"] / "mesh" / "nuc_mesh.h5"
uLagrange = nuc_dict["uLagrange"]
nuc_mesh = uLagrange.function_space().mesh()
mf3_orig = nuc_dict["mf3"]
mf2 = nuc_dict["mf2"]
mf3 = d.MeshFunction("size_t", nuc_mesh, 3, 0)
mf3.array()[:] = mf3_orig.array()[:]
mf3.array()[mf3.array()==1] = 2
mesh_tools.write_mesh(nuc_mesh, mf2, mf3, deformed_mesh_file)
# deformed_mesh_file = pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "spreadCell_mesh.h5"
# deformed_mesh_curv = pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "spreadCell_mesh.h5"
parent_mesh = mesh.ParentMesh(
    mesh_filename=str(deformed_mesh_file),
    mesh_filetype="hdf5",
    name="parent_mesh",
    # curvature=pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "curvatures.xdmf",
    # extra_keys=["subdomain0_2"],
    # key_for_facets="subdomain0_2"
)
mf2 = d.MeshFunction("size_t", parent_mesh.dolfin_mesh, 2, value=0)
hdf5 = d.HDF5File(parent_mesh.dolfin_mesh.mpi_comm(), parent_mesh.mesh_filename, "r")
hdf5.read(mf2, f"/mf2")
hdf5.close()
# assert mf2.mesh().id() == parent_mesh.mf["facets"].mesh().id(),\
#     "Mesh for subdomain does not match"
# # set substrate facets to 11
# parent_mesh.mf["facets"].array()[parent_mesh.mf["facets"].array() > 0] = 11
# # assign PM markers from original mesh
# zero_idx = parent_mesh.mf["facets"].array() == 0
# parent_mesh.mf["facets"].array()[zero_idx] = mf2.array()[zero_idx]

# load cell markers
mf3 = d.MeshFunction("size_t", parent_mesh.dolfin_mesh, 3, value=0)
hdf5 = d.HDF5File(parent_mesh.dolfin_mesh.mpi_comm(), parent_mesh.mesh_filename, "r")
hdf5.read(mf3, f"/mf3")
hdf5.close()

 2026-09-13 15:41:55,541 smart.mesh - INFO - HDF5 mesh, "parent_mesh", successfully loaded from file: /root/scratch/test_h2.5/mesh/nuc_mesh.h5! (mesh.py:277) 
 2026-09-13 15:41:55,542 smart.mesh - INFO - 0 subdomains successfully loaded from file: /root/scratch/test_h2.5/mesh/nuc_mesh.h5! (mesh.py:290) 


In [8]:
mesh_ne = d.create_meshview(mf2, 10)
dx_ne = d.Measure("dx", mesh_ne)
mesh_ne_inner = d.create_meshview(mf2, 12)
dx_ne_inner = d.Measure("dx", mesh_ne_inner)
ref_int =  45.74295 * 2 #d.assemble_mixed(1.0*dx_nm)
proj_NE = d.VectorFunctionSpace(mesh_ne, "P", 1)
proj_NE_scalar = d.FunctionSpace(mesh_ne, "P", 1)
proj_NE_inner = d.VectorFunctionSpace(mesh_ne_inner, "P", 1)
proj_NE_inner_scalar = d.FunctionSpace(mesh_ne_inner, "P", 1)

Fglobal = d.Identity(uLagrange.geometric_dimension()) + d.grad(uLagrange)
Vglobal = d.FunctionSpace(uLagrange.function_space().mesh(), "P", 1)
Vglobal_vec = d.VectorFunctionSpace(uLagrange.function_space().mesh(), "P", 1)
Nref = d.Expression(("x[0]/(sqrt(pow(x[0],2)+pow(x[1],2)+pow(x[2]-z0,2)))",
                    "x[1]/(sqrt(pow(x[0],2)+pow(x[1],2)+pow(x[2]-z0,2)))",
                    "(x[2]-z0)/(sqrt(pow(x[0],2)+pow(x[1],2)+pow(x[2]-z0,2)))"), 
                    z0=4.1+1.04, degree=1)
normals = d.project(Nref, Vglobal_vec)
areaFactorVec = d.det(Fglobal) * d.dot(normals, d.inv(Fglobal))
areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
j_outer = d.Function(proj_NE_scalar)
jvals = j_outer.vector()[:]
jcoords = proj_NE_scalar.tabulate_dof_coordinates()
for i in range(len(jcoords)):
    jvals[i] = areaFactor(jcoords[i])
j_outer.vector().set_local(jvals)
j_outer.vector().apply("insert")
innerScale = (R0-nuc_dict["rthickness"])/R0
j_inner = d.Function(proj_NE_inner_scalar)
jvals = j_inner.vector()[:]
jcoords = proj_NE_inner_scalar.tabulate_dof_coordinates()
for i in range(len(jcoords)):
    jvals[i] = areaFactor(jcoords[i])
j_inner.vector().set_local(jvals)
j_inner.vector().apply("insert")
npc_factor = ref_int / d.assemble_mixed(j_outer*dx_ne)
lamin_factor = ref_int / d.assemble_mixed(j_inner*dx_ne_inner)
sa_nuc_inner = d.assemble_mixed(j_inner*dx_ne_inner)
dx_nuc = d.Measure("dx", uLagrange.function_space().mesh())
Fcur = d.Identity(uLagrange.geometric_dimension()) + d.grad(uLagrange)
vol_nuc = d.assemble_mixed(d.det(Fcur)*dx_nuc)
load_data = np.loadtxt(here / ".." / 'plotting/new_yaptaz_nanopillar_data.csv', skiprows=1, delimiter=',')
avg_NC_ratio = np.average(load_data[:,2]/load_data[:,4])
vol_tot = vol_nuc / avg_NC_ratio # vol_nuc + 437.0 * 2 # multiplied by 2 from prev geo because going from 1/8 to 1/4 geo

In [9]:
FActin = Parameter.mesh_quantity("FActin", FActin0, uM, compartment="NE")
LaminA_tot = Parameter("LaminA_tot", lamin_factor*3500.0*laminAbundance, surf_unit)
volSA_ref = Parameter("volSA_ref", 1.50, um) # matching previous study ~68/45
conversionFactor = (vol_nuc/sa_nuc_inner) * 602.214 # molecules/um^2 per uM (in terms of nuclear surf/vol)

k_fl = Parameter("k_fl", 0.46*602.214*1.50, surf_unit / (uM*sec)) # convert from 1/s assuming vol/SA ratio from prev study (~1.50)
p = Parameter("p", 9.0e-6, kPa / uM**2.6)
C_LaminA = Parameter("C_LaminA", 100.0, kPa)
k_rl = Parameter("k_rl", args["k_phos"], 1 / sec)

k_flVal = k_fl.value * p.value*FActin0**2.6/(C_LaminA.value + p.value*FActin0**2.6)
setPhos = k_rl.value*conversionFactor/(k_rl.value*conversionFactor + k_flVal)
if phiE > 0: # then update stiffness to match initial condition
    Efactor = ((1-phiE) + phiE*laminAbundance*(1-setPhos))/((1-phiE) + phiE*laminAbundance)
    for Eupdate in [nuc_dict["E1scale"],nuc_dict["E2scale"]]:
        Eupdate.vector()[:] *= Efactor
        Eupdate.vector().apply("insert")
    nuc_dict["kRamp"][-1] = 0.0
    nuc_dict = nuc_def.solve_next_nuc_step(nuc_dict)
    nuc_dict["kRamp"][-1] = 0.0

    Fglobal = d.Identity(uLagrange.geometric_dimension()) + d.grad(uLagrange)
    Vglobal = d.FunctionSpace(uLagrange.function_space().mesh(), "P", 1)
    Vglobal_vec = d.VectorFunctionSpace(uLagrange.function_space().mesh(), "P", 1)
    areaFactorVec = d.det(Fglobal) * d.dot(normals, d.inv(Fglobal))
    areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
    normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
    j_outer = d.Function(proj_NE_scalar)
    jvals = j_outer.vector()[:]
    jcoords = proj_NE_scalar.tabulate_dof_coordinates()
    for i in range(len(jcoords)):
        jvals[i] = areaFactor(jcoords[i])
    j_outer.vector().set_local(jvals)
    j_outer.vector().apply("insert")
    innerScale = (R0-nuc_dict["rthickness"])/R0
    j_inner = d.Function(proj_NE_inner_scalar)
    jvals = j_inner.vector()[:]
    jcoords = proj_NE_inner_scalar.tabulate_dof_coordinates()
    for i in range(len(jcoords)):
        jvals[i] = areaFactor(jcoords[i])
    j_inner.vector().set_local(jvals)
    j_inner.vector().apply("insert")
    npc_factor = ref_int / d.assemble_mixed(j_outer*dx_ne)
    lamin_factor = ref_int / d.assemble_mixed(j_inner*dx_ne_inner)
    sa_nuc_inner = d.assemble_mixed(j_inner*dx_ne_inner)
    dx_nuc = d.Measure("dx", uLagrange.function_space().mesh())
    Fcur = d.Identity(uLagrange.geometric_dimension()) + d.grad(uLagrange)
    vol_nuc = d.assemble_mixed(d.det(Fcur)*dx_nuc)
    LaminA_tot = Parameter("LaminA_tot", lamin_factor*3500.0*laminAbundance, surf_unit)

 2026-09-13 15:41:55,848 smart.model_assembly - DEBUG - Initializing parameter FActin as mesh quantity (model_assembly.py:743) 
ROOT -2026-09-13 15:41:55,913 mechanotransduction - INFO - Current force is 0.0 (nuclear_deformation_mixed3d.py:1220)
ROOT -2026-09-13 15:41:55,913 mechanotransduction - INFO - Current pressure is 827.2463470008547 (nuclear_deformation_mixed3d.py:1221)


Solving nonlinear variational problem.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 0: r (abs) = 1.235e+03 (tol = 1.000e-06) r (rel) = 1.000e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 1: r (abs) = 2.475e+05 (tol = 1.000e-06) r (rel) = 2.004e+02 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 2: r (abs) = 3.250e+03 (tol = 1.000e-06) r (rel) = 2.632e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 3: r (abs) = 4.624e+01 (tol = 1.000e-06) r (rel) = 3.745e-02 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
 

ROOT -2026-09-13 15:41:59,024 mechanotransduction - INFO - Current volume is 166.5936555610854 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)
ROOT -2026-09-13 15:41:59,027 mechanotransduction - INFO - Computing idx = 8, outer iteration 1, def = 1.3897707214275297 (nuclear_deformation_mixed3d.py:1320)
ROOT -2026-09-13 15:41:59,028 mechanotransduction - INFO - Current force is 0.0 (nuclear_deformation_mixed3d.py:1220)
ROOT -2026-09-13 15:41:59,028 mechanotransduction - INFO - Current pressure is 825.2463470008547 (nuclear_deformation_mixed3d.py:1221)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 1.209e+06 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 4.027e+03 (tol = 1.000e-08) r (rel) = 3.331e-03 (tol = 1.000e-06)
  Newton iteration 2: r (abs) = 1.055e+01 (tol = 1.000e-08) r (rel) = 8.725e-06 (tol = 1.000e-06)
  Newton iteration 3: r (abs) = 4.244e-04 (tol = 1.000e-08) r (rel) = 3.510e-10 (tol = 1.000e-06)
  Newton solver finished in 3 iterations and 3 linear solver iterations.
Solving nonlinear variational problem.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 0: r (abs) = 3.622e+06 (tol = 1.000e-06) r (rel) = 1.000e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 1: r (abs) = 2.970e+03 (tol = 1.000e-06) r (rel) = 8.200e-04 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain f

ROOT -2026-09-13 15:42:00,500 mechanotransduction - INFO - Current volume is 164.04282834448574 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)
ROOT -2026-09-13 15:42:00,505 mechanotransduction - INFO - Computing idx = 8, outer iteration 2, def = 1.308486160650216 (nuclear_deformation_mixed3d.py:1320)
ROOT -2026-09-13 15:42:00,506 mechanotransduction - INFO - Current force is 0.0 (nuclear_deformation_mixed3d.py:1220)
ROOT -2026-09-13 15:42:00,506 mechanotransduction - INFO - Current pressure is 823.2463470008547 (nuclear_deformation_mixed3d.py:1221)


...

Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 5.078e+04 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 5.421e+00 (tol = 1.000e-08) r (rel) = 1.067e-04 (tol = 1.000e-06)
  Newton iteration 2: r (abs) = 2.456e-05 (tol = 1.000e-08) r (rel) = 4.836e-10 (tol = 1.000e-06)
  Newton solver finished in 2 iterations and 2 linear solver iterations.
Solving nonlinear variational problem.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 0: r (abs) = 3.998e+00 (tol = 1.000e-06) r (rel) = 1.000e+00 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Newton iteration 1: r (abs) = 3.295e+02 (tol = 1.000e-06) r (rel) = 8.241e+01 (tol = 1.000e-06)
  *** Warning: Found no facets matching domain for boundary condition.
  *** Warning: Found no facets matching domain for boundary condition.
  Ne

ROOT -2026-09-13 15:42:34,223 mechanotransduction - INFO - Current volume is 143.81948878624556 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)
ROOT -2026-09-13 15:42:34,227 mechanotransduction - INFO - Done computing idx = 8 after 20 iterations, def = 1.2752618627877048 (nuclear_deformation_mixed3d.py:1322)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 3.920e+04 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 3.246e+00 (tol = 1.000e-08) r (rel) = 8.282e-05 (tol = 1.000e-06)
  Newton iteration 2: r (abs) = 8.922e-06 (tol = 1.000e-08) r (rel) = 2.276e-10 (tol = 1.000e-06)
  Newton solver finished in 2 iterations and 2 linear solver iterations.


ROOT -2026-09-13 15:42:34,648 mechanotransduction - INFO - Current volume is 143.81948878624547 (ref is 61.80048529445672) (nuclear_deformation_mixed3d.py:1050)


Solving nonlinear variational problem.
  Newton iteration 0: r (abs) = 8.922e-06 (tol = 1.000e-08) r (rel) = 1.000e+00 (tol = 1.000e-06)
  Newton iteration 1: r (abs) = 1.573e-09 (tol = 1.000e-08) r (rel) = 1.763e-04 (tol = 1.000e-06)
  Newton solver finished in 1 iterations and 1 linear solver iterations.


Module C: nucleo-cytoplasmic transport

In [10]:
# c3a: LaminA dephos. and phos.
conversionFactor = (vol_nuc/sa_nuc_inner) * 602.214 # molecules/um^2 per uM (in terms of nuclear surf/vol)
setPhos = k_rl.value*conversionFactor/(k_rl.value*conversionFactor + k_flVal)
LaminAp = Species("LaminAp", LaminA_tot.value*setPhos/conversionFactor, vol_unit, 4.0, D_unit, "Nuc")#, 
                #    alt_deform=["0.0","0.0","0.0"], alt_manual_update=True)
LaminA.initial_condition = LaminA_tot.value - LaminAp.initial_condition * conversionFactor

c3a_1 = Reaction(
    "c3a_1",
    ["LaminAp"],
    ["LaminA"],
    param_map={
        "k_fl": "k_fl",
        "p": "p",
        "C_LaminA": "C_LaminA",
        "FActin": "FActin",
    },
    explicit_restriction_to_domain="NE",
    eqn_f_str="LaminAp*k_fl*p*FActin**2.6/(C_LaminA + p*FActin**2.6)",
)
c3a_2 = Reaction(
    "c3a_2",
    ["LaminA"],
    ["LaminAp"],
    param_map={
        "k_rl": "k_rl",
    },
    explicit_restriction_to_domain="NE",
    eqn_f_str="k_rl*LaminA",
)

# # c3b: LaminA disassembly
# kcycle = args["kcycle"]
# phiCycle = args["phi_cycle"] # ratio of assembly to degradation
# kdeg_lamin = Parameter("kdeg_lamin", kcycle/phiCycle, 1/sec)
# # kdeg_lamin = Parameter("kdeg_lamin", 0.0, 1/sec)
# c3b = Reaction(
#     "c3b",
#     ["LaminAp"],
#     [],
#     param_map={
#         "kdeg": "kdeg_lamin",
#     },
#     explicit_restriction_to_domain="NE",
#     eqn_f_str="kdeg*LaminAp",
# )

# # c3c: LaminA reassembly
# kassembly_lamin = Parameter("kassembly_lamin", kcycle/phiCycle, 1/sec)
# # kassembly_lamin = Parameter("kassembly_lamin", 0.0, 1/sec)
# LaminA_set = Parameter("LaminA_set", LaminA_tot.value, surf_unit)
# c3c = Reaction(
#     "c3c",
#     [],
#     ["LaminA"],
#     param_map={
#         "kassembly": "kassembly_lamin",
#         "LaminA_set": "LaminA_set",
#     },
#     explicit_restriction_to_domain="NE",
#     eqn_f_str="kassembly*(LaminA_set - LaminA)",
# )

if laminOnly:
    kdeg_yap = Parameter("kdeg_yap", 0.0, 1/sec)
    YAPTAZ_nuc = Species("YAPTAZ_nuc", 0.7, uM, 4.0, D_unit, "Nuc")
    rYAP = Reaction("rYAP", ["YAPTAZ_nuc"], [], 
                    param_map={"kdeg":"kdeg_yap"},
                    eqn_f_str="YAPTAZ_nuc*kdeg")
else:
    # Module C: nucleo-cytoplasmic transport
    # c1: YAP/TAZ deseq. and sequestration in the cytosol
    # k_CN = Parameter("k_CN", 0.56, 1 / sec)
    # k_CY = Parameter("k_CY", 0.00076, 1 / (sec * uM**2))
    # k_NC = Parameter("k_NC", 0.14, 1 / sec)
    Myo_A = Parameter("Myo_A", 4.0, uM)
    # c1 = Reaction(
    #     "c1",
    #     ["YAPTAZ_seq"],
    #     ["YAPTAZ"],
    #     param_map={"k_CN": "k_CN", "k_CY": "k_CY", 
    #                 "k_NC": "k_NC", "Myo_A": "Myo_A", "FActin": "FActin"},
    #     species_map={
    #         "YAPTAZ": "YAPTAZ",
    #         "YAPTAZ_seq": "YAPTAZ_seq",
    #     },
    #     eqn_f_str="YAPTAZ_seq*(k_CN + k_CY*FActin*Myo_A) - k_NC*YAPTAZ",
    # )

    # c2: nuclear membrane transport
    # eta0val = 3.0
    # eta = Species("eta", eta0val, dimensionless, 0.1, D_unit, "NE")
    # kcycle_nm = Parameter("kcycle_nm", 0.01, 1/sec)
    # eta0 = Parameter("eta0", eta0val, dimensionless)
    # c2 = Reaction("c2", [], ["eta"],
    #                 param_map={"kcycle_nm": "kcycle_nm", "eta0": "eta0"},
    #     explicit_restriction_to_domain="NE",
    #     eqn_f_str="kcycle_nm*(eta0-eta)")
    # c4: opening and closing of NPCs
    # eta0val = 3
    # eta = Parameter("eta", 3.0, dimensionless)
    NPCtot = 6.5*npc_factor
    k_fNPCval = 2.8e-7
    k_rNPCval = 8.7
    K_NPC = k_fNPCval / k_rNPCval
    NPC_A.initial_condition = NPCtot * ((K_NPC*LaminA.initial_condition*FActin0*Myo_A.value)/
                                        (1 + K_NPC*LaminA.initial_condition*FActin0*Myo_A.value))
    NPC = Species("NPC", NPCtot - NPC_A.initial_condition, surf_unit, 0.0, D_unit, "NE")
    # NPC_tot = Parameter.from_expression("NPC_tot", f"(6.5/{eta0val})*{npc_factor}*(1 - {args['npc_slope']}*(z-{zMin})/({zMax-zMin}))", surf_unit)
    k_fNPC = Parameter("k_fNPC", k_fNPCval, 1 / (sec * uM**2 * surf_unit))
    k_rNPC = Parameter("k_rNPC", k_rNPCval, 1 / sec)
    c4 = Reaction(
        "c4",
        ["NPC"],
        ["NPC_A"],
        param_map={"k_fNPC": "k_fNPC", "k_rNPC": "k_rNPC", 
                "Myo_A": "Myo_A", "FActin": "FActin"},
        explicit_restriction_to_domain="NE",
        species_map={
            "LaminA": "LaminA",
        },
        eqn_f_str="NPC*k_fNPC*LaminA*FActin*Myo_A - k_rNPC*NPC_A",
    )

    # c5: nuclear translocation of YAP/TAZ
    Nconv = 602.2
    Ytot = Nconv*(0.9*(vol_tot-vol_nuc) + 0.7*vol_nuc)
    K_CN = 4.0
    K_CY = 5.429e-3
    k_insoloval = 1.0
    k_in2val = 10.0
    k_outval = 1.0
    scaleIn = Parameter("scaleIn", 0.75, dimensionless)
    K_solo = scaleIn.value*k_insoloval/k_outval
    K_2 = scaleIn.value*k_in2val/k_outval
    phiNP = (K_CN + K_CY*FActin0*Myo_A.value)/(1+ K_CN + K_CY*FActin0*Myo_A.value)
    Ynuc0 = (Ytot*phiNP/Nconv) * ((K_solo + K_2*NPC_A.initial_condition)/
                                  ((K_solo + K_2*NPC_A.initial_condition)*phiNP*vol_nuc + (vol_tot-vol_nuc)))
    YAPTAZ_nuc.initial_condition = Ynuc0
    Ycyto = (Ytot - Ynuc0*vol_nuc*Nconv)/((vol_tot-vol_nuc)*Nconv)
    Yfree = Ycyto*phiNP
    k_insolo = Parameter("k_insolo", k_insoloval, surf_unit / (sec * uM))
    k_in2 = Parameter("k_in2", k_in2val, 1 / (sec * uM))
    k_out = Parameter("k_out", k_outval, surf_unit / (sec * uM))
    YAPTAZ = Parameter("YAPTAZ", Yfree, uM)
    if args["a0_npc"] > 0:
        a0_NPC = Parameter("a0_NPC", args["a0_npc"], dimensionless)
        c5 = Reaction(
            "c5",
            [],
            ["YAPTAZ_nuc"],
            param_map={"k_insolo": "k_insolo", "k_in2": "k_in2", "k_out": "k_out", 
                       "a0_NPC": "a0_NPC", "aNPC": "aNPC", "YAPTAZ": "YAPTAZ", "scaleIn": "scaleIn"},
            species_map={"YAPTAZ_nuc": "YAPTAZ_nuc", "NPC_A": "NPC_A"},
            explicit_restriction_to_domain="NE",
            eqn_f_str="scaleIn*YAPTAZ*exp((aNPC-1)/a0_NPC)*(k_insolo + k_in2*NPC_A) - k_out*YAPTAZ_nuc",
        )
    else:
        c5 = Reaction(
            "c5",
            [],
            ["YAPTAZ_nuc"],
            param_map={"k_insolo": "k_insolo", "k_in2": "k_in2", "k_out": "k_out", 
                       "YAPTAZ": "YAPTAZ", "scaleIn": "scaleIn"},
            species_map={"YAPTAZ_nuc": "YAPTAZ_nuc", "NPC_A": "NPC_A"},
            explicit_restriction_to_domain="NE",
            eqn_f_str="scaleIn*YAPTAZ*(k_insolo + k_in2*NPC_A) - k_out*YAPTAZ_nuc",
        )

KeyboardInterrupt: 

In [ ]:
pc, sc, cc, rc = sbmodel_from_locals(locals().values())

In [ ]:
# set config for current run
configCur = config.Config()
configCur.flags.update(
    {
        "allow_unused_components": True,
    }
)
configCur.solver.update(
    {
        "final_t": 10000.0,
        "initial_dt": args["time_step"],
        "time_precision": 6,
        "use_snes": True,
        "attempt_timestep_restart_on_divergence": True,
        "reset_timestep_for_negative_solution": False,
    }
)

In [ ]:
model_cur = model.Model(pc, sc, cc, rc, configCur, parent_mesh)
model_cur.initialize(initialize_solver=True)
# Write initial condition(s) to file
results = dict()
result_folder = args["outdir"]
result_folder.mkdir(exist_ok=True, parents=True)

Initialize deformation fields.

In [ ]:
uglobal = uLagrange #nuc_dict["uLagrangeFull"]
uprev = uglobal.copy(deepcopy=True)
# proj_cyto = cc["Cyto"].deform_func.function_space()
proj_NE = cc["NE"].deform_func.function_space()
proj_Nuc = cc["Nuc"].deform_func.function_space()
# cc["Cyto"].deform_func.assign(d.interpolate(uglobal, proj_cyto))
# cc["Cyto"].deform_prev.assign(d.interpolate(uglobal, proj_cyto))
cc["NE"].deform_func.assign(d.interpolate(uglobal, proj_NE))
cc["NE"].deform_prev.assign(d.interpolate(uglobal, proj_NE))
cc["NE"].deform_func_global.assign(uglobal)
cc["NE"].deform_prev_global.assign(uprev)
cc["Nuc"].deform_func.assign(d.interpolate(uglobal, proj_Nuc))
cc["Nuc"].deform_prev.assign(d.interpolate(uglobal, proj_Nuc))
cc["Nuc"].deform_func_global.assign(uglobal)
cc["Nuc"].deform_prev_global.assign(uprev)

# map inner velocities
uInnerVals = sc["LaminA"].alt_deform_func.vector()[:] #nuc_dict["uLagrangeInner"]
NECoords = sc["LaminA"].alt_deform_func.function_space().tabulate_dof_coordinates()
nucRad1 = nuc_dict["nucRad1"]
nucRad2 = nuc_dict["nucRad2"]
rthickness = nuc_dict["rthickness"]
zthickness = nuc_dict["zthickness"]
z0 = min(NECoords[:,2])
for i in range(0,len(NECoords),3):
    xcur = NECoords[i]
    rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
    if np.abs(rcur - nucRad1) > 0.01*nucRad1:
        raise ValueError("Not at the right point")
    if nucRad1 != nucRad2 or rthickness != zthickness:
        raise ValueError("inner extrapolation does not work for this geo")
    xcur_inner = xcur * (nucRad2-rthickness)/nucRad2
    xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rthickness)/nucRad2
    uInnerVals[i:i+3] = (xcur_inner - xcur) + uglobal(xcur_inner)

for inner_sp in ["LaminA"]:#, "LaminAp"]:
    sc[inner_sp].alt_deform_func.vector().set_local(uInnerVals)
    sc[inner_sp].alt_deform_func.vector().apply("insert")
    sc[inner_sp].alt_deform_prev.vector().set_local(uInnerVals)
    sc[inner_sp].alt_deform_prev.vector().apply("insert")

In [ ]:
# # b7: curvature-dependent actin polymerization at the PM
npSpacing = nuc_args["nanopillar_spacing"]
npRad = nuc_args["nanopillar_radius"]
hNP = nuc_args["nanopillar_height"]
if npSpacing == 0.0:
    xNP = np.array([])
    yNP = np.array([])
else:
    xMax = np.ceil(2*nucRad1 / npSpacing) * npSpacing
    xNP = np.arange(-xMax, xMax+1e-12, npSpacing)
    yNP = np.arange(-xMax, xMax+1e-12, npSpacing)
    xNP, yNP = np.meshgrid(xNP, yNP)
    xNP = xNP.flatten()
    yNP = yNP.flatten()
FActin_func = pc["FActin"].dolfin_function
FActin_coords = FActin_func.function_space().tabulate_dof_coordinates()
Fvals = FActin_func.vector()[:]
dSteric = 0.2
for i in range(len(FActin_coords)):
    xDef = FActin_coords[i] + uglobal(FActin_coords[i])
    if len(xNP)==0: # then flat
        Fvals[i] = FActin0 + actinBoost*np.exp(-(xDef[2]+dSteric)/actinDecay)*np.exp(0.0/2.0)
    else:
        all_dist = np.sqrt((xNP-xDef[0])**2 + (yNP-xDef[1])**2)
        min_idx = np.argmin(all_dist)
        curvTop = 0.0
        curvSide = 1/(2*(npRad+0.05))
        if np.any(all_dist <= npRad+0.05):
            dTop = xDef[2] + dSteric
        else:
            dTop = np.sqrt((all_dist[min_idx]-npRad-0.05)**2 + (xDef[2]+dSteric)**2)
        if xDef[2] > -dSteric:
            dSide = np.sqrt((all_dist[min_idx]-npRad-0.05)**2 + (xDef[2]+dSteric)**2)
        else:
            dSide = np.abs(all_dist[min_idx] - npRad - 0.05)
        curvDep = np.max([np.exp(-dTop/actinDecay)*np.exp(curvTop/2.0),
                          np.exp(-dSide/actinDecay)*np.exp(curvSide/2.0)])
        Fvals[i] = FActin0 + actinBoost*curvDep
FActin_func.vector().set_local(Fvals)
FActin_func.vector().apply("insert")
FActin_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"FActin.xdmf"))
FActin_file.parameters["flush_output"] = True
FActin_file.write(FActin_func, model_cur.t)

Initialize values of nuclear stretch (alpha).

In [ ]:
if args["a0_npc"] > 0:
    avec = nuc_dict["a_vector"]
    # afun = ufl.sqrt(avec[0]**2 + avec[1]**2 + avec[2]**2)
    # amesh = avec.function_space().mesh()
    # Vscalar_a = d.FunctionSpace(avec.function_space().mesh(), "P", 1)
    # assert np.all(d.vertex_to_dof_map(sc["aNPC"].V) == d.vertex_to_dof_map(Vscalar_a))
    # assert np.all(d.vertex_to_dof_map(pc["aNPC"].dolfin_function.function_space()) == d.vertex_to_dof_map(Vscalar_a))
    # aproj = d.project(afun, Vscalar_a)
    # avals = sc["aNPC"].u["u"].vector()[:]
    # avals[sc["aNPC"].dof_map] = aproj.vector()[:]
    # for tag in ["u", "n"]:
    #     sc["aNPC"].u[tag].vector().set_local(avals)
    #     sc["aNPC"].u[tag].vector().apply("insert")
    afunc = pc["aNPC"].dolfin_function
    avals = afunc.vector()[:]
    acoords = afunc.function_space().tabulate_dof_coordinates()
    for i in range(len(acoords)):
        avec_cur = avec(acoords[i])
        avals[i] = np.sqrt(avec_cur[0]**2 + avec_cur[1]**2 + avec_cur[2]**2)
    ainit = np.mean(avals)
    avals /= ainit
    pc["aNPC"].dolfin_function.vector().set_local(avals)
    pc["aNPC"].dolfin_function.vector().apply("insert")
    aNPC_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"aNPC.xdmf"))
    aNPC_file.parameters["flush_output"] = True
    aNPC_file.write(pc["aNPC"].dolfin_function, model_cur.t)

Write out settings used in current simulation.

In [ ]:
if model_cur.mpi_comm_world.rank == 0:
    import json
    # Dump config to results folder
    (result_folder / "config.json").write_text(
        json.dumps(
            {
                "solver": configCur.solver.__dict__,
                "flags": configCur.flags.__dict__,
                "reaction_database": configCur.reaction_database,
                "mesh_file": str(args["mesh_folder"]),
                "outdir": str(args["outdir"]),
                "time_step": args["time_step"],
            }
        )
    )

Initialize XDMF files for storing output.

In [ ]:
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(
        model_cur.mpi_comm_world, str(result_folder / f"{species_name}.xdmf")
    )
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)

Jdict = dict()
for c in model_cur._active_compartments:
    cname = c.name
    results[cname] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{cname}_udef.xdmf"))
    results[cname].parameters["flush_output"] = True
    udef = c.deform_func
    results[cname].write(udef, model_cur.t)
    jname = f"J_{c.name}"
    results[jname] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{jname}.xdmf"))
    results[jname].parameters["flush_output"] = True
    if c.is_volume:
        Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
        Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
        Jdict[cname] = d.project(d.det(Fcur), Vscalar)
        results[jname].write(Jdict[cname], model_cur.t)
        c.jacobian.assign(Jdict[cname])
        c.jacobian_prev.assign(Jdict[cname].copy(deepcopy=True))
    else:
        Fglobal = d.Identity(uglobal.geometric_dimension()) + d.grad(uglobal)
        Vglobal = d.FunctionSpace(uglobal.function_space().mesh(), "P", 1)
        Vglobal_vec = d.VectorFunctionSpace(uglobal.function_space().mesh(), "P", 1)
        areaFactorVec = d.det(Fglobal) * d.dot(cc["NE"].normals, d.inv(Fglobal))
        areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
        normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
        jvals = c.jacobian.vector()[:]
        jcoords = c.jacobian.function_space().tabulate_dof_coordinates()
        for i in range(len(jcoords)):
            jvals[i] = areaFactor(jcoords[i])
        c.jacobian.vector().set_local(jvals)
        c.jacobian.vector().apply("insert")
        c.jacobian_prev.assign(c.jacobian.copy(deepcopy=True))
        results[jname].write(c.jacobian, model_cur.t)
        Vvec = d.VectorFunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
        nvals = cc["NE"].deformed_normals.vector()[:]
        ncoords = cc["NE"].deformed_normals.function_space().tabulate_dof_coordinates()
        for i in range(0,len(ncoords),3):
            ncur = normalVec(ncoords[i])
            nvals[i:i+3] = ncur/np.sqrt(ncur[0]**2 + ncur[1]**2 + ncur[2]**2)
        cc["NE"].deformed_normals.vector().set_local(nvals)
        cc["NE"].deformed_normals.vector().apply("insert")
if sc["LaminA"].alt_deform_func != None:
    c = sc["LaminA"].compartment
    cnamealt = f"{c.name}_alt"
    results[cnamealt] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{cnamealt}_udef.xdmf"))
    results[cnamealt].parameters["flush_output"] = True
    udef = sc["LaminA"].alt_deform_func
    results[cnamealt].write(udef, model_cur.t)
    Fglobal = d.Identity(uglobal.geometric_dimension()) + d.grad(uglobal)
    Vglobal = d.FunctionSpace(uglobal.function_space().mesh(), "P", 1)
    Vglobal_vec = d.VectorFunctionSpace(uglobal.function_space().mesh(), "P", 1)
    areaFactorVec = d.det(Fglobal) * d.dot(cc["NE"].normals, d.inv(Fglobal))
    areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
    normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
    innerScale = (R0-rthickness)/R0
    jvals = sc["LaminA"].alt_jacobian.vector()[:]
    jcoords = sc["LaminA"].alt_jacobian.function_space().tabulate_dof_coordinates()
    for i in range(len(jcoords)):
        jvals[i] = (innerScale**2)*areaFactor(innerScale*jcoords[i,0],
                                              innerScale*jcoords[i,1],
                                              (z0+R0)+innerScale*(jcoords[i,2]-z0-R0))
    sc["LaminA"].alt_jacobian.vector().set_local(jvals)
    sc["LaminA"].alt_jacobian.vector().apply("insert")
    sc["LaminA"].alt_jacobian_prev.assign(sc["LaminA"].alt_jacobian.copy(deepcopy=True))
    # define deformed normals for NE
    Vvec = d.VectorFunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
    nvals = sc["LaminA"].alt_deformed_normals.vector()[:]
    ncoords = sc["LaminA"].alt_deformed_normals.function_space().tabulate_dof_coordinates()
    for i in range(0,len(ncoords),3):
        ncur = normalVec(innerScale*ncoords[i,0],
                         innerScale*ncoords[i,1],
                         (z0+R0)+innerScale*(ncoords[i,2]-z0-R0))
        nvals[i:i+3] = ncur/np.sqrt(ncur[0]**2 + ncur[1]**2 + ncur[2]**2)
    sc["LaminA"].alt_deformed_normals.vector().set_local(nvals)
    sc["LaminA"].alt_deformed_normals.vector().apply("insert")
    # sc["LaminAp"].alt_jacobian.assign(Jcur)

ulin = nuc_dict["ulin"]
Finit = d.Identity(3) + d.grad(ulin)
Vscalar_global = d.FunctionSpace(ulin.function_space().mesh(), "P", 1)
Jinit = d.project(d.det(Finit), Vscalar_global)
Jinit = Jinit.copy(deepcopy=True)
Jcur = d.det(d.Identity(3) + d.grad(ulin))
Escale = nuc_dict["E1scale"]#d.project(nuc_args["softFactor"], Vscalar_global)
sc["LaminA"].sol.set_allow_extrapolation(True)
# sc["LaminAp"].sol.set_allow_extrapolation(True)

In [ ]:
# Set loglevel to warning in order not to pollute notebook output
# smart_logger.setLevel(logging.INFO)

# define integration measures
nucMesh = model_cur.cc["Nuc"].dolfin_mesh
dxNuc = d.Measure("dx", domain=nucMesh)
# int_val = d.assemble_mixed(model_cur.sc["YAPTAZ_nuc"].u["u"] * dxNuc)
# vol_nuc = d.assemble_mixed(1.0 * dxNuc)
# YAPTAZ_nuc_vec = np.array([int_val / vol_nuc])

Solve system over all timesteps. 

In [ ]:
# Solve
force0 = nuc_dict["kRamp"][-1] 
# nuc_dict["kRamp"][-1] = force0 # to avoid initial jump in force
forceMax = args["force_val"]
# Vmarker = d.FunctionSpace(cc["Cyto"].dolfin_mesh, "P", 1)
# cytosolic_marker = d.interpolate(d.Expression("1.0",degree=1), Vmarker)
# marker_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"cytosolic_marker.xdmf"))
# marker_file.parameters["flush_output"f] = True
# marker_file.write(cytosolic_marker, model_cur.t)
# eta_func = model_cur.sc["eta"].u["u"]
t0_deform = args["t0_deform"]
stress_var = nuc_dict["surf_stress"]
stress_var.set_allow_extrapolation(True)
Fcur = d.Identity(3) + d.grad(nuc_dict["ulin"])
Vscalar = d.FunctionSpace(nuc_dict["ulin"].function_space().mesh(), "P", 1)
# W_var = d.project(nuc_dict["psi_fcn"]/d.det(Fcur), Vscalar)
# W_var.set_allow_extrapolation(True)
surf_tension = nuc_dict["surf_tension"]
V_NE = d.FunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
rupture_marker = d.Function(V_NE) #d.project(1/(sc["LaminA"].sol + sc["LaminAp"].sol), V_NE)
rupture_vec = rupture_marker.vector()[:]
rupture_marker2 = d.Function(V_NE)
rupture_vec2 = rupture_marker2.vector()[:]
NEStressCoords = V_NE.tabulate_dof_coordinates()
for i in range(len(NEStressCoords)):
    xcur = NEStressCoords[i]
    rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
    rdiff = 0.99*rthickness/2
    xcur_inner = xcur * (nucRad2-rdiff)/nucRad2
    xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rdiff)/nucRad2
    stressCur = stress_var(xcur_inner)
    rupture_vec[i] = (np.sqrt(stressCur[0]**2 + stressCur[1]**2 + stressCur[2]**2) /
                       (sc["LaminA"].sol(xcur)))# + sc["LaminAp"].sol(xcur)))
    rupture_vec2[i] = surf_tension(xcur_inner) / sc["LaminA"].sol(xcur)
rupture_marker.vector().set_local(rupture_vec)
rupture_marker.vector().apply("insert")
rupture_marker2.vector().set_local(rupture_vec2)
rupture_marker2.vector().apply("insert")
rupture_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"rupture_marker.xdmf"))
rupture_file.parameters["flush_output"] = True
rupture_file.write(rupture_marker, model_cur.t)
rupture_file2 = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"rupture_marker2.xdmf"))
rupture_file2.parameters["flush_output"] = True
rupture_file2.write(rupture_marker2, model_cur.t)
Escale_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"Escale.xdmf"))
Escale_file.parameters["flush_output"] = True
Escale_file.write(Escale, model_cur.t)
dx_NE = d.Measure("dx", domain=cc["NE"].dolfin_mesh)
npc_int = [d.assemble_mixed(cc["NE"].jacobian*(sc["NPC"].sol+sc["NPC_A"].sol)*dx_NE)]
lamin_int = [d.assemble_mixed(sc["LaminA"].alt_jacobian*sc["LaminA"].sol*dx_NE)]
YAPTAZ_NC = [Ynuc0/Ycyto]
while True:
    # for sp in ["NPC", "NPC_A"]:
    #     Dcur = d.project(d.Constant(0.003)/eta_func, sc[sp].V)
    #     sc[sp].D_dolfin.assign(Dcur)
    logger.info("First, update deformation")
    # consider time step restriction due to force jump
    nuc_dict["tvec"] = model_cur.tvec.copy()
    nuc_dict["tvec"].append(model_cur.t+model_cur.dt)
    forceCur = force0 + (forceMax-force0)*(
        1-np.exp(-float(model_cur.t+model_cur.dt)/t0_deform))
    if forceCur - nuc_dict["kRamp"][-2] > 2.0:
        logger.info("Restricting time step due to force jump")
        forceCur = nuc_dict["kRamp"][-2] + 2.0
        tval = -t0_deform*np.log(((forceMax-force0) - (forceCur-force0))/(forceMax-force0))
        model_cur.set_dt(tval - float(model_cur.t))
    if False:#forceCur - nuc_dict["kRamp"][-2] < 0.01 or forceMax == 0.0:
        logger.info("Skipping mechnical calculation because force has not changed much")
    else:
        nuc_dict["kRamp"][-1] = forceCur
        uprev.assign(uglobal)
        nuc_dict = nuc_def.solve_next_nuc_step(nuc_dict)
        uglobal = uLagrange #nuc_dict["uLagrangeFull"]
        # cc["Cyto"].deform_func.assign(d.interpolate(uglobal, proj_cyto))
        Jglobal = d.project(d.det(d.Identity(3) + d.grad(uglobal)), 
                            d.FunctionSpace(uglobal.function_space().mesh(), "P", 1))
        uvals = uglobal.vector()[:]

        # if np.any(Jglobal.vector()[:] <= 0.0):
        #     Jdof = d.vertex_to_dof_map(Jglobal.function_space())
        #     udof = d.vertex_to_dof_map(uglobal.function_space())
        #     for c in d.cells(uglobal.function_space().mesh()):
        #         Jvals = Jglobal.vector()[Jdof[c.entities(0)]]
        #         udof_cur = udof[3*c.entities(0)]
        #         if np.any(Jvals < 0.0):
        #             for dofval in udof_cur:
        #                 uvals[dofval:dofval+3] = (uprev.vector()[dofval:dofval+3] + 
        #                                           uvals[dofval:dofval+3])/2
            # print("You need to do something here.")
            # uglobal.vector().set_local(uvals)
            # uglobal.vector().apply("insert")
            # Jglobal = d.project(d.det(d.Identity(3) + d.grad(uglobal)), 
            #                 d.FunctionSpace(uglobal.function_space().mesh(), "P", 1))
            # if np.any(Jglobal.vector()[:] <= 0.0):
            #     uglobal.vector().set_local(uprev.vector()[:])
            #     uglobal.vector().apply("insert")

        cc["NE"].deform_func.assign(d.interpolate(uglobal, proj_NE))
        cc["NE"].deform_func_global.assign(uglobal)
        cc["Nuc"].deform_func.assign(d.interpolate(uglobal, proj_Nuc))
        cc["Nuc"].deform_func_global.assign(uglobal)
        # map inner velocities
        uInnerVals = sc["LaminA"].alt_deform_func.vector()[:]
        for i in range(0,len(NECoords),3):
            xcur = NECoords[i]
            rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
            xcur_inner = xcur * (nucRad2-rthickness)/nucRad2
            xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rthickness)/nucRad2
            uInnerVals[i:i+3] = (xcur_inner - xcur) + uglobal(xcur_inner)
        for inner_sp in ["LaminA"]:#, "LaminAp"]:
            sc[inner_sp].alt_deform_func.vector().set_local(uInnerVals)
            sc[inner_sp].alt_deform_func.vector().apply("insert")
        for i in range(len(FActin_coords)):
            xDef = FActin_coords[i] + uglobal(FActin_coords[i])
            if len(xNP)==0: # then flat
                Fvals[i] = FActin0 + actinBoost*np.exp(-(xDef[2]+dSteric)/actinDecay)*np.exp(0.0/2.0)
            else:
                all_dist = np.sqrt((xNP-xDef[0])**2 + (yNP-xDef[1])**2)
                min_idx = np.argmin(all_dist)
                curvTop = 0.0
                curvSide = 1/(2*(npRad+0.05))
                if np.any(all_dist <= npRad+0.05):
                    dTop = xDef[2] + dSteric
                else:
                    dTop = np.sqrt((all_dist[min_idx]-npRad-0.05)**2 + (xDef[2]+dSteric)**2)
                if xDef[2] > -dSteric:
                    dSide = np.sqrt((all_dist[min_idx]-npRad-0.05)**2 + (xDef[2]+dSteric)**2)
                else:
                    dSide = np.abs(all_dist[min_idx] - npRad - 0.05)
                curvDep = np.max([np.exp(-dTop/actinDecay)*np.exp(curvTop/2.0),
                                np.exp(-dSide/actinDecay)*np.exp(curvSide/2.0)])
                Fvals[i] = FActin0 + actinBoost*curvDep
        FActin_func.vector().set_local(Fvals)
        FActin_func.vector().apply("insert")
        if args["a0_npc"] > 0:
            avec = nuc_dict["a_vector"]
            for i in range(len(acoords)):
                avec_cur = avec(acoords[i])
                avals[i] = np.sqrt(avec_cur[0]**2 + avec_cur[1]**2 + avec_cur[2]**2)/ainit
            pc["aNPC"].dolfin_function.vector().set_local(avals)
            pc["aNPC"].dolfin_function.vector().apply("insert")
        stress_var = nuc_dict["surf_stress"]
        stress_var.set_allow_extrapolation(True)
        Fcur = d.Identity(3) + d.grad(nuc_dict["ulin"])
        Vscalar = d.FunctionSpace(nuc_dict["ulin"].function_space().mesh(), "P", 1)
        # W_var = d.project(nuc_dict["psi_fcn"]/d.det(Fcur), Vscalar)
        # W_var.set_allow_extrapolation(True)
        surf_tension = nuc_dict["surf_tension"]
        for i in range(len(NEStressCoords)):
            xcur = NEStressCoords[i]
            rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
            xcur_inner = xcur * (nucRad2-rdiff)/nucRad2
            xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rdiff)/nucRad2
            stressCur = stress_var(xcur_inner)
            # rupture_vec[i] = stressCur / (sc["LaminA"].sol(xcur))
            rupture_vec[i] = (np.sqrt(stressCur[0]**2 + stressCur[1]**2 + stressCur[2]**2) /
                            (sc["LaminA"].sol(xcur)))# + sc["LaminAp"].sol(xcur)))
            rupture_vec2[i] = surf_tension(xcur_inner) / sc["LaminA"].sol(xcur)
        rupture_marker.vector().set_local(rupture_vec)
        rupture_marker.vector().apply("insert")
        rupture_marker2.vector().set_local(rupture_vec2)
        rupture_marker2.vector().apply("insert")
    
    if args["a0_npc"] > 0:
        aNPC_file.write(pc["aNPC"].dolfin_function, model_cur.t+model_cur.dt)
    rupture_file.write(rupture_marker, model_cur.t+model_cur.dt)
    rupture_file2.write(rupture_marker2, model_cur.t+model_cur.dt)
    FActin_file.write(FActin_func, model_cur.t+model_cur.dt)

    for c in model_cur._active_compartments:
        cname = c.name
        jname = f"J_{c.name}"
        udef = c.deform_func
        results[cname].write(udef, model_cur.t+model_cur.dt)
        # Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
        # Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
        # Jcur = d.project(d.det(Fcur), Vscalar)
        # Jcur = d.interpolate(Jglobal, Vscalar)
        # if np.any(Jcur.vector()[:] < 0.0):
        #     Jvals = Jcur.vector()[:]
        #     Jvals[Jvals < 0.0] = 0.0
        #     Jcur.vector().set_local(Jvals)
        #     Jcur.vector().apply("insert")
        # c.jacobian.assign(Jcur)
        # Jdict[cname].assign(Jcur)
        # results[jname].write(Jdict[cname], model_cur.t+model_cur.dt)
        if c.is_volume:
            Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
            Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
            Jdict[cname].assign(d.project(d.det(Fcur), Vscalar))
            results[jname].write(Jdict[cname], model_cur.t)
            c.jacobian.assign(Jdict[cname])
        else:
            Fglobal = d.Identity(uglobal.geometric_dimension()) + d.grad(uglobal)
            Vglobal = d.FunctionSpace(uglobal.function_space().mesh(), "P", 1)
            Vglobal_vec = d.VectorFunctionSpace(uglobal.function_space().mesh(), "P", 1)
            areaFactorVec = d.det(Fglobal) * d.dot(cc["NE"].normals, d.inv(Fglobal))
            areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
            normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
            jvals = c.jacobian.vector()[:]
            jcoords = c.jacobian.function_space().tabulate_dof_coordinates()
            for i in range(len(jcoords)):
                jvals[i] = areaFactor(jcoords[i])
            c.jacobian.vector().set_local(jvals)
            c.jacobian.vector().apply("insert")
            results[jname].write(c.jacobian, model_cur.t)
            Vvec = d.VectorFunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
            nvals = cc["NE"].deformed_normals.vector()[:]
            ncoords = cc["NE"].deformed_normals.function_space().tabulate_dof_coordinates()
            for i in range(0,len(ncoords),3):
                ncur = normalVec(ncoords[i])
                nvals[i:i+3] = ncur/np.sqrt(ncur[0]**2 + ncur[1]**2 + ncur[2]**2)
            cc["NE"].deformed_normals.vector().set_local(nvals)
            cc["NE"].deformed_normals.vector().apply("insert")
    if sc["LaminA"].alt_deform_func != None:
        c = sc["LaminA"].compartment
        cnamealt = f"{c.name}_alt"
        udef = sc["LaminA"].alt_deform_func
        results[cnamealt].write(udef, model_cur.t)
        Fglobal = d.Identity(uglobal.geometric_dimension()) + d.grad(uglobal)
        Vglobal = d.FunctionSpace(uglobal.function_space().mesh(), "P", 1)
        Vglobal_vec = d.VectorFunctionSpace(uglobal.function_space().mesh(), "P", 1)
        areaFactorVec = d.det(Fglobal) * d.dot(cc["NE"].normals, d.inv(Fglobal))
        areaFactor = d.project(d.sqrt(d.inner(areaFactorVec, areaFactorVec)), Vglobal)
        normalVec = d.project(areaFactorVec/areaFactor, Vglobal_vec)
        innerScale = (R0-rthickness)/R0
        jvals = sc["LaminA"].alt_jacobian.vector()[:]
        jcoords = sc["LaminA"].alt_jacobian.function_space().tabulate_dof_coordinates()
        for i in range(len(jcoords)):
            jvals[i] = (innerScale**2)*areaFactor(innerScale*jcoords[i,0],
                                                innerScale*jcoords[i,1],
                                                (z0+R0)+innerScale*(jcoords[i,2]-z0-R0))
        sc["LaminA"].alt_jacobian.vector().set_local(jvals)
        sc["LaminA"].alt_jacobian.vector().apply("insert")
        # define deformed normals for NE
        Vvec = d.VectorFunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
        nvals = sc["LaminA"].alt_deformed_normals.vector()[:]
        ncoords = sc["LaminA"].alt_deformed_normals.function_space().tabulate_dof_coordinates()
        for i in range(0,len(ncoords),3):
            ncur = normalVec(innerScale*ncoords[i,0],
                            innerScale*ncoords[i,1],
                            (z0+R0)+innerScale*(ncoords[i,2]-z0-R0))
            nvals[i:i+3] = ncur/np.sqrt(ncur[0]**2 + ncur[1]**2 + ncur[2]**2)
        sc["LaminA"].alt_deformed_normals.vector().set_local(nvals)
        sc["LaminA"].alt_deformed_normals.vector().apply("insert")
        # Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
        # Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
        # Jcur = d.project(d.det(Fcur), Vscalar)
        # Jcur = d.interpolate(Jglobal, Vscalar)
        # if np.any(Jcur.vector()[:] < 0.0):
        #     Jvals = Jcur.vector()[:]
        #     Jvals[Jvals < 0.0] = 0.0
        #     Jcur.vector().set_local(Jvals)
        #     Jcur.vector().apply("insert")
        # sc["LaminA"].alt_jacobian.assign(Jcur)
        # sc["LaminAp"].alt_jacobian.assign(Jcur)
    # J_file.write(Jproj, model_cur.t+model_cur.dt)

    # udef = cc["Cyto"].deform_func
    # Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
    # Jcur = d.det(Fcur)
    # uprev = cc["Cyto"].deform_prev
    # Fprev = d.Identity(uprev.geometric_dimension()) + d.grad(uprev)
    # Jprev = d.det(Fprev)
    # Vscalar = d.FunctionSpace(cc["Cyto"].dolfin_mesh, "P", 1)
    # cytosolic_marker.assign(d.project(cytosolic_marker*Jprev/Jcur, Vscalar))
    # marker_file.write(cytosolic_marker, model_cur.t+model_cur.dt)
    # for sp in cc["Cyto"].species.values():
    #     u_advected = d.project(sp.u["u"]*Jprev/Jcur, Vscalar)
    #     sp.u["u"].assign(u_advected)

    logger.info(f"Solve for time step {model_cur.t}")
    model_cur.monolithic_solve()
    model_cur.adjust_dt()

    # update values of YAP/TAZ
    vol_nuc = d.assemble_mixed(Jdict["Nuc"] * dxNuc)
    Fnuc = d.Identity(cc["Nuc"].deform_func.geometric_dimension()) + d.grad(cc["Nuc"].deform_func)
    phiNP = (K_CN + K_CY*FActin0*Myo_A.value)/(1+ K_CN + K_CY*FActin0*Myo_A.value)
    dxNuc = d.Measure("dx", domain=cc["Nuc"].dolfin_mesh)
    Ycyto = (Ytot - d.assemble_mixed(d.det(Fnuc)*sc["YAPTAZ_nuc"].u["u"]*dxNuc)*Nconv)/((vol_tot-vol_nuc)*Nconv)
    pc["YAPTAZ"].value = Ycyto*phiNP
    pc["YAPTAZ"].value_vector = np.vstack((pc["YAPTAZ"].value_vector, [float(model_cur.t), pc["YAPTAZ"].value]))
    pc["YAPTAZ"].dolfin_constant.assign(pc["YAPTAZ"].value)
    YAPTAZ.dolfin_constant.assign(Ycyto*phiNP)
    np.savetxt(str(result_folder / "YAPTAZ.txt"), pc["YAPTAZ"].value_vector)
    YAPTAZ_NC.append(d.assemble_mixed(d.det(Fnuc)*sc["YAPTAZ_nuc"].u["u"]*dxNuc)/(vol_nuc*Ycyto))
    np.savetxt(str(result_folder / "YAPTAZ_NC.txt"), YAPTAZ_NC)

    # int_val = d.assemble_mixed(Jdict["Nuc"]*model_cur.sc["YAPTAZ_nuc"].u["u"] * dxNuc)
    # vol_nuc = d.assemble_mixed(Jdict["Nuc"] * dxNuc)
    # current_YAPTAZ_nuc = np.array([int_val / vol_nuc])
    # YAPTAZ_nuc_vec = np.concatenate((YAPTAZ_nuc_vec, current_YAPTAZ_nuc))
    
    # adjust Escale
    # ulin = nuc_dict["ulin"]
    # Jcur = d.det(d.Identity(3) + d.grad(ulin))
    # Jratio_ref = d.project(Jinit/Jcur, Vscalar_global)
    Escale_vals = Escale.vector()[:]
    global_coords = Vscalar_global.tabulate_dof_coordinates()
    lamin_coords = sc["LaminA"].V.tabulate_dof_coordinates()
    LaminAdof = sc["LaminA"].dof_map
    # LaminApdof = sc["LaminAp"].dof_map
    for i in range(len(global_coords)):
        xcur = global_coords[i]
        # project to outer NE for lamin calc
        Rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-z0-R0)**2)
        xcur[0] *= (R0/Rcur)
        xcur[1] *= (R0/Rcur)
        xcur[2] = z0 + R0 + (R0/Rcur)*(xcur[2]-z0-R0)
        # try:
        LaminTotCur = sc["LaminA"].sol(xcur)# + sc["LaminAp"].sol(xcur)
        # except:
        #     logger.warning("Could not evaluate lamin at point, using nearest neighbor")
        #     test_dist = np.sqrt((lamin_coords[:,0]-xcur[0])**2 +
        #                         (lamin_coords[:,1]-xcur[1])**2 + 
        #                         (lamin_coords[:,2]-xcur[2])**2)
        #     cur_idx = np.argmin(test_dist)
        #     LaminTotCur = (sc["LaminA"].sol.vector()[LaminAdof[cur_idx]] +
        #                    sc["LaminAp"].sol.vector()[LaminApdof[cur_idx]])
        Escale_vals[i] = (1-phiE) + phiE*laminAbundance*(LaminTotCur/LaminA_tot.value)
    Escale.vector().set_local(Escale_vals)
    Escale.vector().apply("insert")
    nuc_dict["E1scale"].assign(Escale)
    nuc_dict["E2scale"].assign(Escale)
    Escale_file.write(Escale, model_cur.t)

    npc_int.append(d.assemble_mixed(cc["NE"].jacobian*(sc["NPC"].sol+sc["NPC_A"].sol)*dx_NE))
    np.savetxt(result_folder / "npc_int.txt", np.array(npc_int))
    lamin_int.append(d.assemble_mixed(sc["LaminA"].alt_jacobian*sc["LaminA"].sol*dx_NE))
    np.savetxt(result_folder / "lamin_int.txt", np.array(lamin_int))


    # Save results for post processing
    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)
    
    np.savetxt(result_folder / "tvec.txt", np.array(model_cur.tvec).astype(np.float32))

    # if model_cur.mpi_comm_world.rank == 0:
    #     np.savetxt(result_folder / "YAPTAZ_nuc.txt", YAPTAZ_nuc_vec.astype(np.float32))
    # End if we've passed the final time
    if model_cur.t >= model_cur.final_t:
        break